In [ ]:
import torch 
import os, sys
import numpy as np
os.environ["T2struc_NAME"] = "T2struc-15B"
sys.path.append("../../../")
from utils.design_utils import load_pinal

t2struc, text_tokenizer, structre_tokenizer, saprot, saprot_text_tokenizer, saprot_tokenizer = load_pinal()


## Recalculate the NLL of pinalpetase36

In [ ]:
DEVICE="cuda"
@torch.no_grad()
def _CalNLL(desc, InputSeq, InputStruc, t2struc, text_tokenizer, structre_tokenizer, saprot, saprot_text_tokenizer, saprot_tokenizer):
    t2struc_batch = {}

    struc_encode = structre_tokenizer(
        InputStruc,
        return_tensors="pt",
        max_length=1024,
        truncation=True,
        padding="longest",
    )
    t2struc_batch["structure_token_ids"] = struc_encode.input_ids.to(DEVICE)
    t2struc_batch["structure_token_masks"] = struc_encode.attention_mask.to(DEVICE)
    desc_encodings = text_tokenizer(
        desc,
        return_tensors="pt",
        max_length=768,
        truncation=True,
        padding="longest",
    )
    t2struc_batch["text_ids"] = desc_encodings.input_ids.to(DEVICE)
    t2struc_batch["text_masks"] = desc_encodings.attention_mask.to(DEVICE)
    t2struc_batch["text_hidden_states"], t2struc_batch["text_attention_mask"] = t2struc.infer_text(t2struc_batch)
    struc_out = t2struc.infer(t2struc_batch, 
                        text_hidden_states=t2struc_batch["text_hidden_states"],
                        text_attention_mask=t2struc_batch["text_attention_mask"],)
    struct_nll_all = struc_out["outputs"]["loss"] * (len(InputSeq) + 1)
    struct_nll_per_token = struc_out["outputs"]["loss"]
    
    ### sequence
    saprot_input = []
    saprot_input.append("#" + "#".join([i for i in InputStruc if i.isalpha() or i == "#"]))
    inputs = saprot_tokenizer(saprot_input, return_tensors="pt", max_length=1024, truncation=True, padding="longest")
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    ## saprot not mask
    saprot_input_nomask = []
    saprot_input_nomask.append("".join([f"{j}{i}" for j, i in zip(InputSeq, InputStruc) if i.isalpha() or i == "#"]))
    inputs_no_mask = saprot_tokenizer(saprot_input_nomask, return_tensors="pt", max_length=1024, truncation=True, padding="longest")
    inputs_no_mask = {k: v.to(DEVICE) for k, v in inputs_no_mask.items()}
    encodings = saprot_text_tokenizer(
        desc,
        padding="longest", # max_length,longest
        truncation=True,
        max_length=512,
        return_tensors="pt",
    )
    saprot_text_ids = encodings["input_ids"].to(DEVICE)
    saprot_text_masks= encodings["attention_mask"].to(DEVICE)
    SaProtInputDict= {
            "mask_prot_ids": inputs["input_ids"],
            "prot_masks": inputs["attention_mask"],
            'text_ids': saprot_text_ids,
            'text_masks': saprot_text_masks, 
            "prot_ids": inputs_no_mask["input_ids"]
        }
    seq_out = saprot(SaProtInputDict)
    seq_nll_all = seq_out["outputs"]["loss"] * (len(InputSeq) + 2)
    seq_nll_per_token =  seq_out["outputs"]["loss"]

    return {
        "struct_nll_all": struct_nll_all.cpu().item(), 
        "seq_nll_all": seq_nll_all.cpu().item(),
        "struct_nll_per_token": struct_nll_per_token.cpu().item(), 
        "seq_nll_per_token": seq_nll_per_token.cpu().item()}

In [ ]:
desc = "PET hydrolases are a class of enzymes capable of catalyzing the degradation of polyethylene terephthalate (PET), a widely used plastic found in bottles and packaging. This enzyme class includes cutinases, lipases, and esterases, which hydrolyze the ester bonds in PET through a conserved catalytic triad (Ser–His–Asp/Glu). PETase is a well-characterized PET hydrolase that specifically targets and breaks down PET polymers."

seq_dict1 = {"name": "pinalpetase36", "seq": "MAITTGAGPAPTAATLQASSGPYSVTSVSVSSTGASGFGGGTIYYPTASGSFGAVAISPGFTGSQSSIAWLARHLASHGFVVVAINTNSTLDSPQNRGNQLISALDYLTNSAPAAVTAKIDTGRLGVMGHSMGGGGTLIAATQRPDLKAAIPLTPWNLTTNFSGIKAPTLIIGAQNDTIAPVALHSRPFYNSLPSDVPKAYIEITGATHFTPTFPNPTISRYSVAWLKRYVDGDARYDSYLCPGPTPGTNPNISRYKTNNPP", "structure_token": "ddqplddddqddlvqlldlahqfdkdkdwfpcvlqdlfhtfikiagpdqdaaaeeeeeeaqldalllqvslqnrlrrngytytrthgpdshdalcsllssrvsvlvcqcppddpssnsrydnqqyayeyaasnlssqlnnqlvcvsyqegerelyhdpdlascsgqhqyeyehecaeppshcvnhvvsnqvnhdlphwykywyfapdysspssdddslcssvvnlssccgssvncssqvcvvvhddapphrgtndmdtsppd"}

In [ ]:
res = _CalNLL(desc, seq_dict1["seq"], seq_dict1["structure_token"], t2struc, text_tokenizer, structre_tokenizer, saprot, saprot_text_tokenizer, saprot_tokenizer)
nll_per_residue = res["struct_nll_per_token"] + res["seq_nll_per_token"]
ppl = np.exp(nll_per_residue)
nll_per_residue, ppl

## plot the distribution of PPL of all designed petase.

In [ ]:
## Directly compute the PPL values from the saved generation log probabilities to avoid recalculation.
## Note that the loaded values may not be exactly identical to the recalculated values,
## because T2Struct applies logits processors/warpers during generation.
## In practice, this difference is very minor and can be ignored.

In [ ]:
from pathlib import Path
import numpy as np 

seq2ppl = {}
rank_files = Path("../../outputs/petase/rank_files/").glob("res_rank*")
for rank_file in rank_files:
    with open(rank_file, 'r') as f:
        first_line = f.readline()
        for line in f:
            struct, p_1, seq, p_2 = line.strip().split("\t")
            seq_len = len(seq)
            mean_nll_1 = -float(p_1) / (seq_len + 1)
            mean_nll_2 = -float(p_2) / (seq_len + 2)
            seq2ppl[seq] = np.exp(mean_nll_1 + mean_nll_2)
len(seq2ppl)

In [ ]:
ppl_rank_fn = lambda x: sum(1 if x > i else 0 for i in seq2ppl.values()) / len(seq2ppl.values())

ppl_rank_fn(ppl)

In [ ]:
import matplotlib.pyplot as plt

values = seq2ppl.values()

plt.figure(figsize=(8, 5))
plt.hist(values, bins=30)

plt.xlabel("PPL")
plt.ylabel("Count")
plt.title("PPL distribution")

plt.tight_layout()
plt.show()